# Intake: Ahneman with Doyle's DFT descriptors (`ahneman_doyle_dft`)

The same 3,955 Ahneman *et al.* (Science 2018) Buchwald–Hartwig reactions as
the base **`ahneman`** bundle, but with a different representation: the
**120 DFT descriptors that Doyle's random forest was trained on**, rather than
the one-hot encodings and Kraken descriptor sections this project contributes.
Feature sections that add Kraken's ligand chemistry on top of those 120 are
built alongside it.

Two bundles, two names, so they cannot be mixed up:

| bundle | representation | sections |
| --- | --- | --- |
| `datasets/ahneman/` | **ours**: condition one-hots + a Kraken ligand section | `group_ohe`, `selected_2`, …, `pc_scores` |
| `datasets/ahneman_doyle_dft/` (this one) | **Doyle's** 120 published DFT descriptors, alone, with our ligand representations added, or with them swapped in for Doyle's ligand block | `doyle_full_*`, `doyle_cond_*`, `doyle_ligand_cond_ohe` |

Every run tag, zip and results folder made from this bundle starts
`ahneman_doyle_dft_`, never `ahneman_`.

The reactions table is required (section 4) to be **identical, row for row**,
to the base bundle's, so the folds are identical too and every number here sits
directly beside the `ahneman` runs.

Source: [doylelab/rxnpredict](https://github.com/doylelab/rxnpredict). The
Ahneman reshaping code is `../ahneman/rxnprep.py`, imported rather than copied.

In [1]:
from datetime import date
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/ahneman_doyle_dft/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE.parent / "ahneman"))   # the shared Ahneman reshaping code

from gpc import bundle as bundle_mod
from gpc import prep
from gpc.config import (REACTION_FAMILIES, REACTION_SECTIONS, read_json,
                        validate_config, write_json, write_yaml)
from gpc.data import load_bundle
from gpc.features import _preprocessor, feature_frame, model_columns

import rxnprep as R

pd.set_option("display.width", 220, "display.max_columns", 80, "display.max_colwidth", 80)

# ---- Editable -----------------------------------------------------------
DATASET      = "ahneman_doyle_dft"
DISPLAY_NAME = "Ahneman (Doyle DFT)"
TARGET       = "yield"
GROUP        = "ligand"
CATEGORICAL  = ["aryl_halide", "base", "additive"]   # kept in reactions.csv; not model inputs here
SOURCE_URL   = "https://github.com/doylelab/rxnpredict"
MAPPING      = {"XPhos": 1, "t-BuXPhos": 90, "t-BuBrettPhos": 89, "AdBrettPhos": 347}

RAW      = HERE / "raw"                     # download cache (git-ignored)
REFRESH  = False                            # True re-downloads from GitHub
BASE     = ROOT / "datasets" / "ahneman" / "inputs"   # the base bundle, for the identity check
OUT      = HERE / "inputs"
# --------------------------------------------------------------------------
print("hub:", ROOT)

hub: C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub


## 1. Fetch the Doyle repository files

Straight from GitHub into `raw/`, then compared with the hashes recorded in the
base `ahneman` bundle's `audit.json` when it was built.

Two of the recorded hashes (`data_table.csv`, `kraken_identifiers.csv`) were
taken from a copy whose line endings had been converted to Windows CRLF — the
recorded sizes are larger by exactly one byte per line. So the comparison is
made twice: on the raw bytes, and on the content with line endings normalised.
The second must match for every file; that is the statement "upstream has not
changed since the base bundle was built".

In [2]:
import hashlib

def crlf_sha256(path):
    # The hash this file would have after a CRLF conversion, as the base bundle saw it.
    text = path.read_bytes().replace(b"\r\n", b"\n").replace(b"\n", b"\r\n")
    return hashlib.sha256(text).hexdigest()

sources = R.download_sources(RAW, refresh=REFRESH)
recorded = read_json(BASE / "audit.json")["sources"]
comparison = pd.DataFrame([
    {"file": name, "bytes": rec["bytes"], "recorded bytes": recorded[name]["bytes"],
     "sha256": rec["sha256"][:12] + "...",
     "bytes identical": rec["sha256"] == recorded[name]["sha256"],
     "content identical": recorded[name]["sha256"] in (rec["sha256"], crlf_sha256(RAW / name))}
    for name, rec in sources.items()])
display(comparison)
assert comparison["content identical"].all(), \
    "upstream content differs from what the base ahneman bundle was built from"

,file,bytes,recorded bytes,sha256,bytes identical,content identical
0,data_table.csv,1246868,1251468,fe310b50897e...,False,True
1,ligand.csv,3348,3348,1b96791f1378...,True,True
2,base.csv,418,418,3b3bac99c4dd...,True,True
3,aryl_halide.csv,3864,3864,0f979fd7e875...,True,True
4,additive.csv,4006,4006,3d9ba693e77b...,True,True
5,ligand-list.csv,426,426,19c78fede9a3...,True,True
6,output_table.csv,3201887,3201887,ffbfa7de0ec1...,True,True
7,kraken_identifiers.csv,245844,247297,6d59540fe2a5...,False,True


## 2. What is in the CSVs

`data_table.csv` is the plate record: one row per well with the four component
names, SMILES and yield. The four per-component tables in `R/` hold the DFT
descriptors (Spartan, computed by Doyle's group), one row per molecule, keyed by
`name`. `R/output_table.csv` is those four tables expanded to the full
4 × 3 × 15 × 22 = 3,960-cell grid, with **no key columns** — it is the matrix
`analysis.R` loads and models.

In [3]:
raw = {n: pd.read_csv(RAW / f"{n}.csv") for n in
       ["data_table", "additive", "aryl_halide", "base", "ligand", "output_table"]}

rows = []
for name, frame in raw.items():
    if name in R.COMPONENTS:
        n_desc = frame.shape[1] - 1
    elif name == "output_table":
        n_desc = frame.shape[1]
    else:
        n_desc = ""
    rows.append({"file": f"{name}.csv", "rows": len(frame), "columns": frame.shape[1],
                 "key": "name" if "name" in frame else "(none)", "descriptors": n_desc})
display(pd.DataFrame(rows))

# Descriptor families per component: atom-centred NMR shifts and electrostatic
# charges (the '*' columns), vibrational modes, and whole-molecule properties.
def family(column):
    if "_NMR_shift" in column:
        return "NMR shift (atom)"
    if "_electrostatic_charge" in column:
        return "electrostatic charge (atom)"
    if "_frequency" in column or "_intensity" in column:
        return "vibration"
    return "molecular"

families = pd.DataFrame([{"component": comp, "family": family(c)}
                         for comp in ["additive", "aryl_halide", "base", "ligand"]
                         for c in raw[comp].columns if c != "name"])
table = families.value_counts().unstack(fill_value=0)
table["total"] = table.sum(axis=1)
display(table)
for comp in R.COMPONENTS:
    print(f"{comp:12s} {len(raw[comp]):2d} molecules: {', '.join(raw[comp]['name'])}")

,file,rows,columns,key,descriptors
0,data_table.csv,4599,17,(none),
1,additive.csv,22,20,name,19
2,aryl_halide.csv,15,28,name,27
3,base.csv,3,11,name,10
4,ligand.csv,4,65,name,64
5,output_table.csv,3960,120,(none),120


family,NMR shift (atom),electrostatic charge (atom),molecular,vibration,total
component,,,,,
additive,3,5,9,2,19
aryl_halide,6,6,9,6,27
base,0,1,9,0,10
ligand,21,22,1,20,64


ligand        4 molecules: XPhos, t-BuXPhos, t-BuBrettPhos, AdBrettPhos
base          3 molecules: P2Et, BTMG, MTBD
aryl_halide  15 molecules: 1-chloro-4-(trifluoromethyl)benzene, 1-bromo-4-(trifluoromethyl)benzene, 1-iodo-4-(trifluoromethyl)benzene, 1-chloro-4-methoxybenzene, 1-bromo-4-methoxybenzene, 1-iodo-4-methoxybenzene, 1-chloro-4-ethylbenzene, 1-bromo-4-ethylbenzene, 1-ethyl-4-iodobenzene, 2-chloropyridine, 2-bromopyridine, 2-iodopyridine, 3-chloropyridine, 3-bromopyridine, 3-iodopyridine
additive     22 molecules: 5-phenylisoxazole, ethyl-3-methylisoxazole-5-carboxylate, ethyl-5-methylisoxazole-3-carboxylate, 4-phenylisoxazole, 3-phenylisoxazole, 3-methylisoxazole, 5-methylisoxazole, benzo[c]isoxazole, 3,5-dimethylisoxazole, methyl-isoxazole-5-carboxylate, ethyl-isoxazole-3-carboxylate, ethyl-5-methylisoxazole-4-carboxylate, ethyl-isoxazole-4-carboxylate, benzo[d]isoxazole, ethyl-3-methoxyisoxazole-5-carboxylate, 3-methyl-5-phenylisoxazole, N,N-dibenzylisoxazol-3-amine, methyl

## 3. What Doyle's random forest was given

From [`analysis.R`](https://github.com/doylelab/rxnpredict/blob/master/analysis.R):

```r
# Remove reactions without additive and reactions with additive 7
# Remove reactions without aryl halide
output.table  <- read.csv("R\\output_table.csv", header=TRUE)
output.scaled <- as.data.frame(scale(output.table))      # z-score every column
output.scaled$yield <- yield_data                         # attached BY POSITION
output.scaled <- output.scaled[!(is.na(output.scaled$yield)), ]
...
rfFit <- train(yield ~ ., data=training.scaled, trControl=train_control,
               method="rf", importance=TRUE)
```

So the RF input is **every one of the 120 columns of `output_table.csv`**,
z-scored, and nothing else — no one-hot block, no identities. "Additive 7" is
5-phenyl-1,2,4-oxadiazole, which has no row in `additive.csv`; together with the
no-additive and no-aryl-halide control wells those are exactly the exclusions
the base bundle makes, which is why both arrive at 3,955 reactions.

Two differences here, both deliberate:

* `scale()` there is fitted on all 3,960 grid rows at once. Here the
  standardisation is fitted **inside each training fold**, like every other
  section in the hub, so a held-out ligand never informs its own scaling.
* Yields are attached by **name join**, not by row position. The check below
  proves the name join reproduces the published table exactly.

In [4]:
check = R.verify_against_doyle(RAW)
display(pd.Series(check).to_frame("result"))
assert check["same_columns"] and check["identical_row_multiset"], \
    "the component join does not reproduce Doyle's output_table.csv"

published = raw["output_table"]
per_component = {c: sum(col.startswith(c + "_") for col in published.columns)
                 for c in ["additive", "aryl_halide", "base", "ligand"]}
print("published columns:", published.shape[1], per_component)
assert published.shape[1] == 120 == sum(per_component.values())
std = published.std()
print("constant columns (R's scale() would turn these into NaN):", int((std == 0).sum()))
assert (std > 0).all()

,result
published_shape,"[3960, 120]"
rebuilt_shape,"[3960, 120]"
same_columns,True
identical_row_multiset,True


published columns: 120 {'additive': 19, 'aryl_halide': 27, 'base': 10, 'ligand': 64}
constant columns (R's scale() would turn these into NaN): 0


## 4. Reactions — identical to the base `ahneman` bundle

Built by the same `rxnprep.build_reactions`, then compared cell for cell with
`datasets/ahneman/inputs/reactions.csv`. Same rows in the same order means
`make_folds` produces the same folds for every method and seed, so a
`doyle_full` LOLO fold holds out exactly the rows the `group_ohe` LOLO fold did.

In [5]:
reactions, audit = R.build_reactions(RAW)
audit["sources"] = sources
reactions = prep.attach_kraken(reactions.drop(columns=["kraken_id"], errors="ignore"),
                               GROUP, MAPPING)

base = pd.read_csv(BASE / "reactions.csv", keep_default_na=False)
pd.testing.assert_frame_equal(reactions.reset_index(drop=True),
                              base[reactions.columns].reset_index(drop=True),
                              check_dtype=False)
print(f"{len(reactions)} reactions, identical to the base bundle "
      f"(columns: {', '.join(reactions.columns)})")
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

3955 reactions, identical to the base bundle (columns: row_id, ligand, base, aryl_halide, additive, yield, plate, row, col, kraken_id)


,ligand,kraken_id,n,mean,sd,min,max
0,AdBrettPhos,347,990,33.553143,28.390854,0.0,99.034462
1,XPhos,1,990,20.165760,17.534956,0.0,81.357592
2,t-BuBrettPhos,89,989,37.562423,28.717973,0.0,99.999990
3,t-BuXPhos,90,986,41.096600,28.146060,0.0,99.999990


## 5. The reaction descriptor table

`rxnprep.build_rxnpredict_features` joins the four component tables onto each
reaction by name. The columns are put back into **Doyle's own order** (that of
`output_table.csv`), and every reaction's 120-vector is checked to be a row of
the published table.

The `distinct values` column is worth reading: a ligand descriptor takes only 4
values across the whole screen, a base descriptor 3. Under LOLO the held-out
ligand's 64 values are, by construction, values the model never saw — the same
extrapolation question the Kraken sections ask, posed in Doyle's descriptors.

In [6]:
features = R.build_rxnpredict_features(reactions, RAW)
order = list(published.columns)
features = features[["row_id"] + order]
assert list(features.row_id) == list(reactions.row_id)

grid_rows = set(map(tuple, np.round(published[order].to_numpy(float), 8)))
mine = list(map(tuple, np.round(features[order].to_numpy(float), 8)))
assert all(r in grid_rows for r in mine), "a reaction's descriptors are not a published row"
print(f"{features.shape[0]} reactions x {len(order)} descriptors; every row is a row of "
      f"output_table.csv")

component = [next(p for p in ["aryl_halide", "additive", "base", "ligand"] if c.startswith(p))
             for c in order]
summary = pd.DataFrame({"component": component,
                        "distinct": [features[c].nunique() for c in order]})
display(summary.groupby("component").agg(columns=("distinct", "size"),
                                         distinct_values=("distinct", "max")))

3955 reactions x 120 descriptors; every row is a row of output_table.csv

,columns,distinct_values
component,,
additive,19,22
aryl_halide,27,15
base,10,3
ligand,64,4


## 6. The feature sections

Doyle's 120 columns split into two blocks by prefix: the **ligand block**
(`ligand_*`, 64) and the **condition block** (`additive_*` + `aryl_halide_*` +
`base_*`, 56). Our ligand representations are the hub's standard ones — ligand
one-hot, `selected_2`, the two `vbur_*_vmin` pairs, `selected_5`, `pc_top`,
`pc_scores` — plus `kraken_all`, all 190 raw Kraken descriptors.

| family | Doyle columns | + ligand | + condition one-hot | question it answers |
| --- | --- | --- | --- | --- |
| `doyle_full` | all 120 | — | — | the published RF input, exactly |
| `doyle_full_<x>` (**added**) | all 120 | ours, `x` | — | does our ligand chemistry add to theirs? |
| `doyle_cond_<x>` (**in place**) | 56 condition | ours, `x` | — | is ours a substitute for their 64 ligand descriptors? |
| `doyle_ligand_cond_ohe` | 64 ligand | — | aryl halide, base, additive | their ligand chemistry with our condition encoding |

Every `doyle_cond_<x>` has an added twin `doyle_full_<x>` with the same `x`, so
"added vs in place" is a paired comparison, section by section. And
`doyle_cond_<x>` sits beside the base bundle's own `<x>` (same ligand
representation, but one-hot conditions), so the base `ahneman` run, this
in-place family and the added family describe the conditions three ways with
the ligand held fixed.

In [7]:
cfg = prep.bundle_config(
    dataset=DATASET, display_name=DISPLAY_NAME, target=TARGET, group=GROUP,
    categorical=CATEGORICAL, reactions=reactions, source=SOURCE_URL,
    models=list(REACTION_SECTIONS),
    methods=read_json(BASE / "config.json")["evaluation"]["methods"],  # same as the base bundle
    extra={"base_bundle": "ahneman",
           "representation": "Doyle's 120 published DFT descriptors (the RF input): alone, "
                             "with our ligand representations added (doyle_full_*), with them "
                             "in place of Doyle's 64 ligand descriptors (doyle_cond_*), and "
                             "Doyle's ligand block with condition one-hots (doyle_ligand_cond_ohe)"})
cfg = prep.declare_reaction_features(
    cfg, features, name="doyle_dft",
    source="https://github.com/doylelab/rxnpredict/blob/master/R/output_table.csv",
    description="additive (19) + aryl halide (27) + base (10) + ligand (64) DFT descriptors, "
                "joined per reaction by component name; columns in output_table.csv order")

reference = prep.reference_pca()
doyle_ligand = {c for c in order if c.startswith(f"{GROUP}_")}
sections = []
for model in cfg["features"]["models"]:
    numeric, categorical = model_columns(model, cfg, reference)
    doyle = [c for c in numeric if c in set(order)]
    kraken = [c for c in numeric if c not in set(order)]
    one_hot = int(sum(reactions[c].nunique() for c in categorical))
    sections.append({
        "family": next(f for f, ms in REACTION_FAMILIES.items() if model in ms),
        "section": model,
        "doyle ligand": sum(c in doyle_ligand for c in doyle),
        "doyle condition": sum(c not in doyle_ligand for c in doyle),
        "kraken": len(kraken), "one-hot": one_hot,
        "one-hot fields": ", ".join(categorical),
        "encoded": len(numeric) + one_hot})
sections = pd.DataFrame(sections)
display(sections)

# The comparison rests on these: in place carries none of Doyle's ligand
# descriptors, added carries all of them, and every in-place section has a twin.
in_place = sections[sections.family == "in_place"]
assert (in_place["doyle ligand"] == 0).all() and (in_place["doyle condition"] == 56).all()
added = sections[sections.family == "added"]
assert (added["doyle ligand"] == 64).all() and (added["doyle condition"] == 56).all()
assert {m.replace("doyle_cond_", "doyle_full_") for m in in_place.section} <= set(added.section)
print(f"{len(sections)} sections; methods: {cfg['evaluation']['methods']}")

,family,section,doyle ligand,doyle condition,kraken,one-hot,one-hot fields,encoded
0,added,doyle_full,64,56,0,0,,120
1,added,doyle_full_group_ohe,64,56,0,4,ligand,124
2,added,doyle_full_selected_2,64,56,2,0,,122
3,added,doyle_full_vbur_min_vmin,64,56,2,0,,122
4,added,doyle_full_vbur_boltz_vmin,64,56,2,0,,122
5,added,doyle_full_selected_5,64,56,5,0,,125
6,added,doyle_full_pc_top,64,56,12,0,,132
7,added,doyle_full_pc_scores,64,56,4,0,,124
8,added,doyle_full_kraken_all,64,56,190,0,,310
9,in_place,doyle_cond_group_ohe,0,56,0,4,ligand,60


18 sections; methods: ['lolo', 'iid_stratified_4', 'kfold_stratified_5', 'kfold_5']


## 7. Write the bundle, then read it back through the engine

`write_bundle` writes the usual files plus `reaction_features.csv`, after the
reaction-table checks (row alignment, finite, no constant column, no name
clash). Then every section is encoded through the trainer's own
`feature_frame` + `_preprocessor`, so a section that would fail on the cluster
fails here.

In [8]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING, audit=audit,
                           reaction_features=features)

back, ligands, reference, prepared = load_bundle(bundle)
encoded = []
for model in prepared["features"]["models"]:
    X = feature_frame(back, ligands, model, prepared, reference)
    Z = _preprocessor(X, model, prepared, reference).fit_transform(X)
    assert np.isfinite(Z).all()
    encoded.append({"section": model, "rows": Z.shape[0], "encoded inputs": Z.shape[1]})
encoded = pd.DataFrame(encoded)
assert (encoded["encoded inputs"].to_numpy() == sections["encoded"].to_numpy()).all()
display(encoded)
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))

bundle -> C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\datasets\ahneman_doyle_dft\inputs
  3955 rows, 4 ligands, 4 descriptor rows


,section,rows,encoded inputs
0,doyle_full,3955,120
1,doyle_full_group_ohe,3955,124
2,doyle_full_selected_2,3955,122
3,doyle_full_vbur_min_vmin,3955,122
4,doyle_full_vbur_boltz_vmin,3955,122
5,doyle_full_selected_5,3955,125
6,doyle_full_pc_top,3955,132
7,doyle_full_pc_scores,3955,124
8,doyle_full_kraken_all,3955,310
9,doyle_cond_group_ohe,3955,60


,file,KB
0,audit.json,2.9
1,config.json,6.1
2,ligand_features.csv,13.0
3,ligand_mapping.csv,0.4
4,pca_reference.json,6.0
5,pca_reference.npz,12.1
6,reaction_features.csv,3172.1
7,reactions.csv,405.0


## 8. Run config and the HPC zip

Every GP and cluster setting is copied from the base bundle's last run config
(`configs/ahneman_vmin_20260924.json`), so the only difference from those runs
is the representation. 18 sections × 4 methods = **72 jobs**.

The run tag — and so the zip, the cluster's `runs/` folder and the results
folder — is `ahneman_doyle_dft_v2_<date>`. `v2` marks the version with the
in-place and Doyle-ligand families; the earlier 8-section build was
`ahneman_doyle_dft_<date>`, and the different tag keeps the two runs from
writing into one `runs/` folder on the cluster.

In [9]:
# ---- Editable -----------------------------------------------------------
TEMPLATE = ROOT / "configs" / "ahneman_vmin_20260924.json"
RUN_TAG  = f"{DATASET}_v2_{date.today():%Y%m%d}"
# --------------------------------------------------------------------------

template = read_json(TEMPLATE)
run_config = {
    "schema_version": 2,
    "dataset": DATASET,
    "run_tag": RUN_TAG,
    "paths": {"bundle": f"datasets/{DATASET}/inputs", "runs": f"runs/{RUN_TAG}"},
    "run": {**template["run"], "models": list(prepared["features"]["models"]),
            "methods": list(prepared["evaluation"]["methods"]), "feature_columns": {}},
    "gp": dict(template["gp"]),
    "hpc": dict(template["hpc"]),
}
validate_config(run_config)
bundle_mod.check_methods(run_config["run"]["methods"], prepared)

config_path = ROOT / "configs" / f"{RUN_TAG}.json"
write_json(config_path, run_config)
try:
    write_yaml(config_path.with_suffix(".yaml"), run_config)
except ImportError:
    pass

zip_path = bundle_mod.write_zip(run_config, config_path=config_path)

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    contents = pd.DataFrame([{"file": i.filename.split("/", 1)[1],
                              "KB": round(i.file_size / 1024, 1)}
                             for i in archive.infolist()])
display(contents[contents.file.str.startswith(("inputs/", "configs/"))])
print(f"{len(contents)} files, {zip_path.stat().st_size / 1024:,.0f} KB -> upload {zip_path}")

C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\exports\ahneman_doyle_dft_v2_20261001.zip  (227 KB, 39 files)
  run tag   ahneman_doyle_dft_v2_20261001
  tasks     72 (18 sections x 4 methods)
  walltime  00:20:00 per task on gpu:l40:1


,file,KB
28,inputs/config.json,6.1
29,inputs/reactions.csv,405.0
30,inputs/ligand_features.csv,13.0
31,inputs/pca_reference.json,6.0
32,inputs/pca_reference.npz,12.1
33,inputs/reaction_features.csv,3172.1
34,inputs/ligand_mapping.csv,0.4
35,configs/default.json,1.7
36,configs/ahneman_doyle_dft_v2_20261001.json,1.7


39 files, 227 KB -> upload C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\exports\ahneman_doyle_dft_v2_20261001.zip


## 9. When the run comes back

Bring back `runs/ahneman_doyle_dft_v2_<date>/` into the hub, then

```bash
python -m gpc --bundle datasets/ahneman_doyle_dft/inputs \
              --runs runs/ahneman_doyle_dft_v2_<date> report
```

The reactions and folds are the base bundle's, so these numbers can be set
directly beside `runs/ahneman_vmin_20260924/`. The comparisons this run is
built for, all with the folds held fixed:

* **added vs in place** — `doyle_full_<x>` against `doyle_cond_<x>`, for each `x`:
  does keeping Doyle's 64 ligand descriptors help once ours are present?
* **added vs Doyle alone** — `doyle_full_<x>` against `doyle_full`: what our
  ligand chemistry adds on top of the published input.
* **their ligand vs ours** — `doyle_ligand_cond_ohe` against the base bundle's
  `selected_2` … `pc_scores` (same one-hot conditions, different ligand
  descriptors).
* **condition encoding** — base bundle `<x>` (one-hot conditions) against
  `doyle_cond_<x>` (Doyle's condition descriptors), same ligand `x`.

`gpc.config.REACTION_FAMILIES` lists the sections of each family for grouping
them in a figure.